# Scope definition and data verification

## 1. Imports and parameters

Column names are normalized to uppercase so files from different years can be processed consistently.

In [13]:
from pathlib import Path
import pandas as pd

DATA_FOLDER = Path("dados")
MUNICIPALITIES = ["BRAGA", "GUIMARAES", "GUARDA", "FARO"]


## 2. Reading and combining files

CSV and Excel files are searched for in the `dados` folder and its subfolders. Each file is read separately and the tables are concatenated at the end.

In [14]:

files = sorted(
    file
    for extension in ("csv",)
    for file in DATA_FOLDER.glob(f"*.{extension}")
)

if not files:
    raise FileNotFoundError(f"No files found in folder '{DATA_FOLDER}'.")

tables = []

for file in files:
    print("Reading:", file)

    if file.suffix.lower() == ".csv":
        table = pd.read_csv(
            file,
            sep=";",
            decimal=",",
            encoding="utf-8-sig",
            low_memory=False
        )
    else:
        table = pd.read_excel(file)

    table.columns = table.columns.astype(str).str.upper().str.strip()
    tables.append(table)

df = pd.concat(tables, ignore_index=True)

print(f"Total records: {len(df)}")
print("Columns:", list(df.columns))


Reading: dados\Registos_Incendios_SGIF_2001_2010.csv
Reading: dados\Registos_Incendios_SGIF_2011_2020.csv
Reading: dados\Registos_Incendios_SGIF_2021_2025.csv
Total records: 497478
Columns: ['CODIGO_SGIF', 'CODIGO_ANEPC', 'ANO', 'MES', 'DIA', 'HORA', 'AREAPOV_HA', 'AREAMATO_HA', 'AREAAGRIC_HA', 'AREATOTAL_HA', 'CLASSEAREA', 'DATAHORAALERTA', 'DATAHORA_PRIMEIRAINTERVENCAO', 'DATAHORA_EXTINCAO', 'DURACAO_HORAS', 'INCSUP24HORAS', 'DTCCFR', 'DISTRITO', 'CONCELHO', 'FREGUESIA', 'LOCAL', 'RNAP', 'RNMPF', 'X_MILITAR', 'Y_MILITAR', 'LATITUDE', 'LONGITUDE', 'X_ETRS89', 'Y_ETRS89', 'DSR', 'FWI', 'ISI', 'DC', 'DMC', 'FFMC', 'BUI', 'CODCAUSA', 'TIPOCAUSA', 'GRUPOCAUSA', 'DESCRICAOCAUSA', 'FONTEALERTA']


## 3. Municipality normalization and scope definition

Normalization removes whitespace, converts text to uppercase, and removes accents. Only the four municipalities in the scope are then kept.

In [15]:
# Standardize municipality names by converting to uppercase, trimming spaces,
# and removing accents/special characters to create a consistent matching key.
df["NORMALIZED_MUNICIPALITY"] = (
    df["CONCELHO"].astype(str).str.upper().str.strip()
    .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
)

# Identify the municipality whose normalized versions
# match the reference municipality scope and display each unique matches.
matching_spellings = df.loc[
    df["NORMALIZED_MUNICIPALITY"].isin(MUNICIPALITIES),
    "CONCELHO"
].unique()
print("Matching spellings:", matching_spellings)

# Filter records belonging to the municipalities scope
# and create an independent copy of the resulting subset.
scoped_data = df[df["NORMALIZED_MUNICIPALITY"].isin(MUNICIPALITIES)].copy()
print(f"Records in scope: {len(scoped_data)}")


Matching spellings: <ArrowStringArray>
['Guimarães', 'Guarda', 'Braga', 'Faro']
Length: 4, dtype: str
Records in scope: 15534


## 4. Creating the analysis year

When `DATAHORAALERTA` exists, the year is extracted from the alert date. Otherwise, the `ANO` column is used.

In [16]:
# Create a unified year variable from the alert date when available;
# otherwise, use the existing year column.
if "DATAHORAALERTA" in scoped_data.columns:
    scoped_data["ANALYSIS_YEAR"] = pd.to_datetime(
        scoped_data["DATAHORAALERTA"], errors="coerce", dayfirst=True
    ).dt.year
else:
    scoped_data["ANALYSIS_YEAR"] = scoped_data["ANO"]

# Preview the municipality and analysis year fields.
scoped_data[
    [
        "NORMALIZED_MUNICIPALITY",
        "ANALYSIS_YEAR"
    ]
].head() #.head() #.tail()

,NORMALIZED_MUNICIPALITY,ANALYSIS_YEAR
104,GUIMARAES,2001
117,GUIMARAES,2001
127,GUIMARAES,2001
128,GUARDA,2001
165,GUIMARAES,2001


## 5. Cause normalization

Empty values, `nan`, and `None` are treated as missing information. `MISSING_CAUSE` enables those records to be counted and `NORMALIZED_CAUSE` creates an explicit category.

In [17]:
# Clean and standardize cause descriptions, flag missing values,
# and assign a default label to records without cause information.
cause = (
    scoped_data["TIPOCAUSA"].astype(str).str.strip()
    .replace({"": None, "nan": None, "None": None})
)
scoped_data["MISSING_CAUSE"] = cause.isna()
scoped_data["NORMALIZED_CAUSE"] = cause.fillna("SEM INFO (vazio)")

# Preview the original, normalized, and missing-cause indicators.
scoped_data[
    [
        "TIPOCAUSA", 
        "NORMALIZED_CAUSE", 
        "MISSING_CAUSE"
    ]
].head()

,TIPOCAUSA,NORMALIZED_CAUSE,MISSING_CAUSE
104,NaN,SEM INFO (vazio),True
117,NaN,SEM INFO (vazio),True
127,NaN,SEM INFO (vazio),True
128,NaN,SEM INFO (vazio),True
165,NaN,SEM INFO (vazio),True


## 6. Derived variables for analysis

The time between the alert and the first intervention is calculated, and years are grouped into the periods used by the test.

In [18]:
alert_time = pd.to_datetime(scoped_data["DATAHORAALERTA"], errors="coerce", dayfirst=True)
intervention_time = pd.to_datetime(
    scoped_data["DATAHORA_PRIMEIRAINTERVENCAO"], errors="coerce", dayfirst=True
)
scoped_data["INTERVENTION_TIME_MINUTES"] = (intervention_time - alert_time).dt.total_seconds() / 60

scoped_data["PERIOD"] = pd.cut(
    scoped_data["ANALYSIS_YEAR"], bins=[2000, 2016, 2017, 2025],
    labels=["2001-2016", "2017", "2018-2025"]
)

scoped_data[["ANALYSIS_YEAR", "INTERVENTION_TIME_MINUTES", "PERIOD"]].head()

,ANALYSIS_YEAR,INTERVENTION_TIME_MINUTES,PERIOD
104,2001,NaN,2001-2016
117,2001,NaN,2001-2016
127,2001,NaN,2001-2016
128,2001,NaN,2001-2016
165,2001,NaN,2001-2016


## 7. Verification tables

These tables reproduce the script checks: records, causes, percentage without a cause, missing values, intervention times, and the final test table.

In [19]:
print("1) Records by year and municipality")
display(pd.crosstab(scoped_data["ANALYSIS_YEAR"], scoped_data["NORMALIZED_MUNICIPALITY"]))

1) Records by year and municipality


NORMALIZED_MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
ANALYSIS_YEAR,,,,
2001,285,16,159,833
2002,279,15,154,795
2003,381,14,122,950
2004,253,27,74,593
2005,406,29,137,876
2006,330,69,102,434
2007,216,80,84,383
2008,83,42,142,170
2009,242,61,109,487


In [20]:
print("2) Cause type by municipality")
display(pd.crosstab(scoped_data["NORMALIZED_CAUSE"], scoped_data["NORMALIZED_MUNICIPALITY"]))


2) Cause type by municipality


NORMALIZED_MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
NORMALIZED_CAUSE,,,,
Desconhecida,958,163,93,903
Intencional,146,42,230,202
Natural,0,0,50,0
Negligente,168,248,635,369
Reacendimento,278,5,68,160
SEM INFO (vazio),2729,350,730,7007


In [21]:
print("3) % of records without a cause, by year and municipality")
display((pd.crosstab(
    scoped_data["ANALYSIS_YEAR"], scoped_data["NORMALIZED_MUNICIPALITY"], values=scoped_data["MISSING_CAUSE"], aggfunc="mean"
) * 100).round(1))


3) % of records without a cause, by year and municipality


NORMALIZED_MUNICIPALITY,BRAGA,FARO,GUARDA,GUIMARAES
ANALYSIS_YEAR,,,,
2001,98.2,100.0,100.0,99.8
2002,98.9,66.7,87.7,99.9
2003,99.2,100.0,87.7,99.8
2004,99.6,74.1,87.8,99.5
2005,97.3,100.0,92.7,99.2
2006,97.3,95.7,78.4,99.5
2007,18.1,78.8,41.7,93.0
2008,16.9,69.0,9.9,81.8
2009,9.5,86.9,2.8,85.0


In [22]:
columns = [
    "AREATOTAL_HA", "DATAHORAALERTA",
    "DATAHORA_PRIMEIRAINTERVENCAO", "DATAHORA_EXTINCAO", "DURACAO_HORAS",
]
columns = [column for column in columns if column in scoped_data.columns]

print("4) % missing by year")
display((scoped_data.groupby("ANALYSIS_YEAR")[columns].apply(lambda x: x.isna().mean()) * 100).round(1))


4) % missing by year


,AREATOTAL_HA,DATAHORAALERTA,DATAHORA_PRIMEIRAINTERVENCAO,DATAHORA_EXTINCAO,DURACAO_HORAS
ANALYSIS_YEAR,,,,,
2001,0.0,0.0,100.0,0.0,0.0
2002,0.0,0.0,100.0,0.0,0.0
2003,0.0,0.0,100.0,0.0,0.0
2004,0.0,0.0,100.0,0.0,0.0
2005,0.0,0.0,100.0,0.0,0.0
2006,0.0,0.0,100.0,0.0,0.0
2007,0.0,0.0,100.0,0.0,0.0
2008,0.0,0.0,17.8,0.0,0.0
2009,0.0,0.0,12.3,0.0,0.0


In [23]:
print("5) Time to first intervention (minutes)")
display(scoped_data.groupby("NORMALIZED_MUNICIPALITY")["INTERVENTION_TIME_MINUTES"].describe().round(1))
print("Negative:", int((scoped_data["INTERVENTION_TIME_MINUTES"] < 0).sum()),
      "| Above 24h:", int((scoped_data["INTERVENTION_TIME_MINUTES"] > 1440).sum()))


5) Time to first intervention (minutes)


,count,mean,std,min,25%,50%,75%,max
NORMALIZED_MUNICIPALITY,,,,,,,,
BRAGA,2068.0,12.1,7.8,0.0,8.0,11.0,15.0,139.0
FARO,550.0,10.4,5.0,1.0,7.0,10.0,12.0,42.0
GUARDA,915.0,16.2,15.8,0.0,9.0,13.0,20.0,265.0
GUIMARAES,3671.0,12.1,13.4,0.0,7.0,10.0,15.0,615.0


Negative: 0 | Above 24h: 0


In [24]:

hypothesis_data = scoped_data[
    scoped_data["NORMALIZED_CAUSE"].str.lower().isin(["negligente", "intencional"])
    & (scoped_data["PERIOD"] != "2017")
]
print("6) Negligent vs intentional by period (2017 excluded)")
display(pd.crosstab([hypothesis_data["NORMALIZED_MUNICIPALITY"], hypothesis_data["PERIOD"]], hypothesis_data["NORMALIZED_CAUSE"]))


6) Negligent vs intentional by period (2017 excluded)


NORMALIZED_CAUSE                   Intencional  Negligente
NORMALIZED_MUNICIPALITY PERIOD                            
BRAGA                   2001-2016           60         113
                        2018-2025           85          48
FARO                    2001-2016            9         113
                        2018-2025           33         117
GUARDA                  2001-2016          137         454
                        2018-2025           66         147
GUIMARAES               2001-2016           35         145
                        2018-2025          167         221